# 이륙에서 무릎 통과까지 체간 기울기가 얼마나 달라졌을까?

같은 영상에서 **이륙**과 **무릎 통과**를 나란히 보고, 카메라에 가까운 어깨와 골반을 직접 찍습니다. 먼저 바의 위치로 두 시점을 정하고, 그다음 몸통의 기울기를 봅니다.

처음에는 **Run All**을 누르세요. 아래에서 영상 하나를 골라 두 시점을 확인하고 네 점을 찍은 뒤, **마지막 셀만 다시 실행**하면 됩니다. 판독은 자동 저장되므로 한 편씩 나누어 진행해도 됩니다.

## 1. 검증 영상 목록

기존 사람별 검증 분할에서 1차 무릎 통과 라벨(`transition`)이 있는 영상을 불러옵니다. 표의 `탐색 시작`은 `ready` 프레임입니다. 이륙은 다음 단계에서 원본을 보고 별도로 확정합니다. 무릎 통과 라벨이 없거나 가까운 쪽이 미지정인 영상은 목록에서 제외합니다.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

from wlcv.annotation.storage import load_events
from wlcv.annotation.trunk import trunk_pair_editor
from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS
from wlcv.evaluation.trunk import plot_trunk_pair, review_trunk

split = pd.read_csv(SPLIT)
validation = split.loc[split["role"] == "validation", ["video_id", "person_id"]]
metadata = pd.read_csv(ROOT / "data/metadata/videos.csv")
validation = validation.merge(metadata[["video_id", "filename", "near_side"]], on="video_id",
                              how="left", validate="one_to_one")

rows = []
for item in validation.to_dict("records"):
    events = load_events(ANNOTATIONS, item["video_id"], pass_number=1)
    if "transition" in events and item["near_side"] in ("left", "right"):
        rows.append({**item, "탐색 시작": events.get("ready", 0), "통과 라벨": events["transition"]})
if not rows:
    raise ValueError("무릎 통과 라벨과 가까운 쪽 정보가 있는 검증 영상을 확인하세요.")

videos = pd.DataFrame(rows).set_index("video_id")
folder = ANNOTATIONS / "trunk_inclination"
print(f"검증 영상 {len(validation)}편 중 판독 대상 {len(videos)}편")
display(videos[["person_id", "near_side", "탐색 시작", "통과 라벨"]])

## 2. 두 시점을 확인하고 관절 찍기

1. **왼쪽 이륙** 프레임을 앞뒤로 넘겨 원판이 바닥에서 처음 떨어지는 순간을 찾으세요. **오른쪽 무릎 통과**는 바와 무릎의 위치로 확인하세요. 몸통이 얼마나 세워졌는지를 보고 프레임을 고르면 평가할 값이 시점 선택에 영향을 줍니다. 영상 시작부터 바가 떠 있거나 가림 때문에 시점을 정할 수 없으면 **시점 판독 불가**를 누르세요.
2. **두 시점 확인**을 누르면 같은 두 원본에 점을 찍을 수 있습니다. 양쪽 화면에서 선수 기준 가까운 **어깨(견봉)**와 **골반(고관절 소켓의 화면상 위치)**을 차례로 선택하세요.
3. 클릭 전에 **찍을 점**에서 가시성을 고르세요. 옷을 보고 골반 위치를 짐작했다면 **옷에 가려 추정**을 선택합니다. 골반의 기본값은 **소켓 위치 추정**입니다. 보이는 허리선이나 옷의 봉제선을 소켓으로 대신 찍지 마세요. 어깨도 짐작했다면 추정으로 남기고, 지점을 정하기 어렵다면 **가려서 판독 불가** 또는 **위치 불확실**을 누르세요. 판독 상태를 고쳤다면 해당 점을 다시 클릭해 저장하세요.

각 영상의 두 시점과 관절 좌표는 `data/annotations/trunk_inclination/`에 저장됩니다. 영상이나 시점을 다시 열면 그 프레임에서 저장한 점을 이어서 검수할 수 있습니다.

In [ ]:
video_picker = widgets.Dropdown(options=list(videos.index), description="검증 영상")

def open_video(video):
    info = videos.loc[video]
    display(trunk_pair_editor(VIDEOS / info.filename, info, folder))

editor = widgets.interactive_output(open_video, {"video": video_picker})
display(video_picker, editor)

## 3. 각도와 두 프레임 비교하기

골반에서 어깨로 향하는 선이 **화면의 위쪽 수직선**과 이루는 각도를 원본 픽셀 좌표로 계산합니다. 수직이면 0°입니다. **변화량 = 무릎 통과 각도 − 이륙 각도**입니다. 음수면 두 번째 프레임에서 몸통이 더 수직에 가까워졌고, 양수면 수직에서 더 기울었습니다. 촬영 방향이 달라도 기울기의 크기를 비교하도록 좌우 부호는 쓰지 않습니다.

아래 셀을 다시 실행하면 저장된 모든 영상의 판독 상태와 현재 선택한 영상의 원본 두 장이 표시됩니다. 흰 점선은 수직 기준선입니다. 청록 실선과 십자는 직접 확인, 청록 파선과 마름모는 추정이 포함됐음을 표시합니다. 네 점 중 하나라도 추정이면 **참고 변화**에만 값을 넣고, 시점이나 관절을 판독할 수 없으면 변화량을 비워 둡니다. 두 프레임만으로는 그 사이 내내 같은 각도를 유지했는지 알 수 없습니다.

In [ ]:
rows = []
selected = video_picker.value
for video, info in tqdm(videos.iterrows(), total=len(videos), desc="체간 판독 읽기", unit="편"):
    row, points = review_trunk(video, info, folder)
    rows.append(row)
    if video == selected and points:
        plot_trunk_pair(VIDEOS / info.filename, row, points)
        plt.show()

table = pd.DataFrame(rows)
display(table.style.format(precision=1, na_rep="—"))
if any(table["판독"] != "시점 미확인"):
    output_dir = ROOT / "outputs/pose_baseline"
    output_dir.mkdir(parents=True, exist_ok=True)
    table.to_csv(output_dir / "trunk_inclination.csv", index=False)
    print("저장 위치: outputs/pose_baseline/trunk_inclination.csv")
else:
    print("영상에서 두 시점을 먼저 확인하세요.")